#### **0.1 Mounting Google Drive**

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/bone-fracture-detection'
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### **0.2 Pulling from GitHub**

In [9]:
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
os.environ['GH_TOKEN'] = token

username = "baotle"
repo_name = "BoneFractureDetection_Computervision_Project"

!git remote set-url origin https://github.com/{username}/{repo_name}.git
!git -c credential.helper='!f() { echo "username=x-access-token"; echo "password=$GH_TOKEN"; }; f' pull origin main


From https://github.com/baotle/BoneFractureDetection_Computervision_Project
 * branch            main       -> FETCH_HEAD
Already up to date.


#### **0.3 Pushing to GitHub (to be reused)**

In [10]:
token = userdata.get('GITHUB_TOKEN')
os.environ['GH_TOKEN'] = token

username = "baotle"
repo_name = "BoneFractureDetection_Computervision_Project"

!git remote set-url origin https://github.com/{username}/{repo_name}.git
!git add .
!git commit -m "{commit_message}"
!git -c credential.helper='!f() { echo "username=x-access-token"; echo "password=$GH_TOKEN"; }; f' push -q

print("✅ Pushed.")

[main 8816abf] {commit_message}
 2 files changed, 2 insertions(+), 2 deletions(-)
 rewrite notebooks/nb02_training.ipynb (77%)
✅ Pushed.


#### **0.4 Install torch and check for device =GPU***

In [11]:
import torch
print(f"GPU available : {torch.cuda.is_available()}")
print(f"Device name : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only - check Runtime > Change runtime type'}")

GPU available : False
Device name : CPU only - check Runtime > Change runtime type


In [12]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 4.3 MB/s eta 0:00:00


#### ****0.5 Setup Processed Directory****


In [18]:
import os, shutil

DEST = '/content/data/raw'
PROCESSED_DEST = '/content/data/processed'
os.makedirs(DEST, exist_ok=True)
os.makedirs(PROCESSED_DEST, exist_ok=True)

# Regenerate raw data (import your rebuild function or redefine it here)
success = rebuild_dataset_verified(DEST)
print(f"Raw data ready: {success}")

# Regenerate the bbox-converted version
for split in ['train', 'valid', 'test']:
    n = convert_split_to_bbox(DEST, PROCESSED_DEST, split)
    print(f"{split}: converted {n} label files")

shutil.copy2(os.path.join(DEST, 'data.yaml'), os.path.join(PROCESSED_DEST, 'data.yaml'))
print("data.yaml copied")

NameError: name 'rebuild_dataset_verified' is not defined

In [17]:
from ultralytics import YOLO
PROCESSED_DEST = '/content/data/processed'
model = YOLO('yolov8n.pt')

#Using YOLOs inbuilt training functionality
results = model.train(
    data = os.path.join(PROCESSED_DEST, 'data.yaml'),
    epochs = 50,
    imgsz = 640,
    batch = 16,
    project = os.path.join (PROJECT_DIR, 'models'),
    name = 'baseline_yolov8n',

    hsv_h = 0.0, hsv_s = 0.0,
    hsv_v = 0.2,
    degrees = 5,
    translate = 0.1, scale =0.2,
    shear = 0.0, flipud = 0.0,
    fliplr = 0.5,
    mosaic = 0.0
)

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/processed/data.yaml, degrees=5, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=baseline_yolov8n, nbs=64, nms=None, opset=None

RuntimeError: Dataset '/content/data/processed/data.yaml' error ❌ '/content/data/processed/data.yaml' does not exist